# 6a. Optogenetic Inactivation Predictions

**Purpose**: Before running opto experiments, generate model-specific
predictions for what PPC inactivation should do to behaviour.

**Design**: Two opto windows (between-subjects batches):
- **Choice window** (stimulus → response): suppresses decision computation
- **Update window** (outcome → ITI): suppresses belief updating

**Key question**: What does "inactivate PPC" mean computationally?
This notebook explores multiple possibilities and generates predictions
for each.

**Lesion models** (to be tested):

| Lesion model | Choice window effect | Update window effect |
|:------------|:---------------------|:--------------------|
| Null         | Random choice (P=0.5) | No update (η=0) |
| Noise        | Add noise to percept (↑σ) | Add noise to update |
| Attenuation  | Flatten psychometric (↓slope) | Reduce learning rate |
| Prior-only   | Ignore stimulus, use prior | Normal update |

The **null** model is the strongest prediction; the others explore
partial impairment. 30% interleaved trials means we compare opto-on
vs opto-off within the same session.

**Depends on**: 3c (model assignments), 4a (shift dynamics),
5 (SLDS state definitions).

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from models.BE_core import BEParams, BEModel
from models.SC_core import SCParams, SCModel
from analysis.stimulus_distributions import sample_distribution
from behav_utils.analysis.summary_stats import compute_summary_stats
from behav_utils.analysis.update_matrix import compute_update_matrix
from behav_utils.plotting.update_matrix import (
    plot_update_matrix, plot_phase_update_matrices,
)
from behav_utils.plotting.styles import apply_style

apply_style()
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────
TRIALS_PER_SESSION = 300
BURN_IN = 1000
OPTO_FRACTION = 0.30  # fraction of trials with opto
SEED = 42

# Representative expert parameters
BE_EXPERT = BEParams(
    sigma_percep=0.15, A_repulsion=0.10,
    eta_learning=0.10, eta_relax=0.12,  # low η = expert
)
SC_EXPERT = SCParams(
    sigma_percep=0.15, A_repulsion=0.10,
    gamma=0.98, sigma_update=0.30,  # high γ = expert
)

# Post-shift parameters (higher learning rate)
BE_POSTSHIFT = BEParams(
    sigma_percep=0.15, A_repulsion=0.10,
    eta_learning=0.40, eta_relax=0.12,  # high η = adapting
)
SC_POSTSHIFT = SCParams(
    sigma_percep=0.15, A_repulsion=0.10,
    gamma=0.85, sigma_update=0.30,  # low γ = adapting
)

DISTRIBUTIONS = ['Uniform', 'Hard-A']

## 1. Define Lesion Models

Each lesion model is a function that modifies the simulation at the
trial level. The opto mask determines which trials are affected.

In [ ]:
def simulate_with_opto(
    model_type, params, stimuli, categories, opto_mask,
    lesion_type='null', lesion_target='choice',
    burn_in=BURN_IN, seed=SEED,
):
    """
    Simulate a session with trial-level opto inactivation.

    Args:
        model_type: 'BE' or 'SC'
        params: model parameters
        stimuli, categories: trial arrays
        opto_mask: boolean array (True = opto on)
        lesion_type: 'null', 'noise', 'attenuation'
        lesion_target: 'choice' or 'update'
        burn_in: burn-in trials
        seed: random seed

    Returns:
        choices: simulated choices (with opto effects)
        choices_control: what choices would have been without opto
    """
    rng = np.random.default_rng(seed)
    n_trials = len(stimuli)

    # Initialise
    if model_type == 'BE':
        state = BEModel.create_initial_state(
            burn_in=burn_in, params=params, seed=seed)
    else:
        state = SCModel.create_initial_state(
            params=params, burn_in=burn_in, seed=seed)

    # Control: full simulation without opto
    rng_ctrl = np.random.default_rng(seed)
    if model_type == 'BE':
        state_ctrl = BEModel.create_initial_state(
            burn_in=burn_in, params=params, seed=seed)
        choices_ctrl, _, _, _ = BEModel.simulate_session(
            params, state_ctrl, stimuli, categories, rng_ctrl)
    else:
        state_ctrl = SCModel.create_initial_state(
            params=params, burn_in=burn_in, seed=seed)
        choices_ctrl, _, _, _ = SCModel.simulate_session(
            params, state_ctrl, stimuli, categories, rng_ctrl)

    # Opto simulation: trial-by-trial with lesion applied
    # TODO: This requires modifying the trial loop inside
    # BEModel/SCModel. Current simulate_session doesn't support
    # per-trial parameter modification.
    #
    # Options:
    # 1. Post-hoc: run normal simulation, then replace opto trial
    #    choices with lesioned choices. Doesn't capture downstream
    #    effects (opto trial feedback affects future non-opto trials).
    #
    # 2. Modify model to accept per-trial opto flags. More accurate
    #    but requires changes to BE_core/SC_core.
    #
    # For now, use option 1 (post-hoc replacement) as approximation.

    choices_opto = choices_ctrl.copy()

    if lesion_target == 'choice':
        if lesion_type == 'null':
            # Random choice on opto trials
            choices_opto[opto_mask] = rng.choice([0, 1], size=opto_mask.sum())
        elif lesion_type == 'attenuation':
            # Flatten toward 0.5: replace with biased coin flip
            # P(B) for control, pushed toward 0.5
            # Approximation: 50% chance of flipping the choice
            flip = rng.random(opto_mask.sum()) < 0.3
            choices_opto[opto_mask] = np.where(
                flip, 1 - choices_ctrl[opto_mask], choices_ctrl[opto_mask])

    elif lesion_target == 'update':
        if lesion_type == 'null':
            # No update on opto trials: behaviour unchanged on those
            # trials, but the model doesn't learn from them.
            # Post-hoc approximation: the choice is normal, but we mark
            # these trials as if the animal didn't receive feedback.
            # Effect shows up in SUBSEQUENT trials, not the opto trial itself.
            pass  # TODO: requires trial-loop modification

    return choices_opto, choices_ctrl

print('simulate_with_opto defined')
print('NOTE: update-window lesion requires model modification — '
      'currently approximated post-hoc')

## 2. Generate Opto Predictions

For each condition, compute predicted:
- Psychometric curve (opto-on vs opto-off trials)
- Update matrix (opto-on vs opto-off)
- Accuracy difference

In [ ]:
conditions = [
    ('Expert Uniform', 'Uniform', BE_EXPERT, SC_EXPERT),
    ('Post-shift Hard-A', 'Hard-A', BE_POSTSHIFT, SC_POSTSHIFT),
]

lesion_configs = [
    ('choice', 'null',        'Choice: random'),
    ('choice', 'attenuation', 'Choice: attenuated'),
    # ('update', 'null',        'Update: suppressed'),  # TODO
]

all_predictions = []

for cond_name, distribution, be_params, sc_params in conditions:
    rng = np.random.default_rng(SEED)
    stimuli, categories = sample_distribution(
        TRIALS_PER_SESSION * 5,  # pool 5 sessions for power
        distribution, rng=rng,
    )
    opto_mask = rng.random(len(stimuli)) < OPTO_FRACTION

    for model_type, params in [('BE', be_params), ('SC', sc_params)]:
        for target, ltype, desc in lesion_configs:
            choices_opto, choices_ctrl = simulate_with_opto(
                model_type, params, stimuli, categories,
                opto_mask, lesion_type=ltype, lesion_target=target,
            )

            # Compare opto-on vs opto-off
            valid_ctrl = ~np.isnan(choices_ctrl)
            acc_ctrl = np.nanmean(choices_ctrl[valid_ctrl] == categories[valid_ctrl])

            opto_on = opto_mask & ~np.isnan(choices_opto)
            opto_off = ~opto_mask & ~np.isnan(choices_opto)

            acc_on = np.nanmean(choices_opto[opto_on] == categories[opto_on])
            acc_off = np.nanmean(choices_opto[opto_off] == categories[opto_off])

            all_predictions.append({
                'condition': cond_name,
                'model': model_type,
                'lesion': desc,
                'acc_control': acc_ctrl,
                'acc_opto_on': acc_on,
                'acc_opto_off': acc_off,
                'acc_effect': acc_on - acc_off,
            })

pred_df = pd.DataFrame(all_predictions)
print(pred_df.to_string(index=False))

## 3. Prediction Summary Table

The core prediction: if PPC is necessary for updating, then:
- **Choice-window inactivation**: impairs performance at ALL phases
  (PPC is always involved in the decision)
- **Update-window inactivation**: impairs RECOVERY post-shift
  (prevents model updating) but has no effect during expert phase
  (model is already adequate, no updating needed)

In [ ]:
# ── Qualitative prediction table ─────────────────────────────────────────
print('\n=== QUALITATIVE PREDICTIONS ===')
print()
print('If PPC is necessary for model updating (core hypothesis):')
print()
print(f'{"Phase":>20s}  {"Choice window":>20s}  {"Update window":>20s}')
print('-' * 65)
print(f'{"Expert (adequate)":>20s}  {"Impaired":>20s}  {"No effect":>20s}')
print(f'{"Early post-shift":>20s}  {"Impaired":>20s}  {"Severely impaired":>20s}')
print(f'{"Late post-shift":>20s}  {"Impaired":>20s}  {"Reduced effect":>20s}')
print()
print('If PPC is necessary for choice (alternative):')
print()
print(f'{"Phase":>20s}  {"Choice window":>20s}  {"Update window":>20s}')
print('-' * 65)
print(f'{"Expert (adequate)":>20s}  {"Impaired":>20s}  {"No effect":>20s}')
print(f'{"Early post-shift":>20s}  {"Impaired":>20s}  {"No effect":>20s}')
print(f'{"Late post-shift":>20s}  {"Impaired":>20s}  {"No effect":>20s}')
print()
print('The DISCRIMINATING prediction is the update window during')
print('early post-shift: only the updating hypothesis predicts impairment.')

## 4. Design Note: Trial-Level Update-Window Lesion

The current implementation uses post-hoc replacement for choice-window
lesions, which is a reasonable approximation (the choice is replaced
independently of the model state).

For **update-window** lesions, post-hoc replacement is inadequate:
the effect is on the belief update, which cascades to future trials.
This requires modifying `BEModel.simulate_session` and
`SCModel.simulate_session` to accept a per-trial `update_mask`:

```python
# In the trial loop:
if update_mask is not None and not update_mask[t]:
    # Skip belief update for this trial
    pass
else:
    # Normal belief update
    state = update_belief(state, s_hat, reward, params)
```

**TODO**: Add `update_mask` parameter to both model cores.
Then re-run this notebook with proper cascading effects.

## Interpretation

This notebook generates predictions *before* running opto experiments.
The predictions should be registered (written down, shared with
supervisor) before seeing opto data.

**Key result to look for**: Is the update-window effect during
early post-shift larger than during expert? If yes → PPC is
dynamically necessary for model updating. If no → PPC may be
always necessary (for choice) or never necessary (for updating).

**Model-specific predictions**: If the animal is classified as BE
(from 3c), the choice-window lesion should flatten the psychometric
curve. If SC, the effect may manifest differently (category
distributions rather than boundary belief). These differences
are secondary predictions — the primary prediction (update-window
× learning phase interaction) is model-agnostic.